In [ ]:
# Copyright 2025 DeepMind Technologies Limited. All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

[![Colab で開く](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google/genai-processors/blob/main/notebooks/constrained_decoding_intro.ipynb)


# 構造化の明瞭さ: データクラスをモダリティとして扱う

大規模言語モデルはテキスト生成が得意ですが、Python の `dataclasses` のような構造化データを扱いたい場合はどうすればよいでしょうか。GenAI Processors ライブラリは、あなたのカスタムデータ型をテキストや画像と同様に、もう一つのモダリティとして扱います。

これにより、構造化データを取り込んだり生成したり変換したりするパイプラインを簡単に作成できます。主なポイントは以下のとおりです:

*   **モダリティとしての Dataclass:** `ProcessorPart.from_dataclass()` と `part.get_dataclass()` を使えば、カスタム `dataclasses` を `ProcessorPart` に簡単に詰めたり取り出したりできます。内部表現は単なる JSON です。
*   **モデルとの自動連携:** `GenaiModel` や `OllamaModel` では、`response_schema`（例: `response_schema=MyDataclass`）を指定するだけで、制約付きデコーディングのリクエスト送信からモデルの JSON 出力のパース、型付き `ProcessorPart` への変換までを自動処理します。スキーマがリストの場合は各アイテムが個別のパートとして順次出力され、並行処理が可能になります。

このノートブックでは、これらの機能を段階的に紹介し、構造化データを AI ワークフローへシームレスに統合する方法を示します。


## 1. ⚙️ セットアップ

まず、GenAI Processors ライブラリと依存関係をインストールしましょう。


In [ ]:
!pip install -q genai-processors

### API キー

GenAI のモデルプロセッサを使用するには Gemini の API キーが必要です。まだお持ちでない場合は Google AI Studio から取得し、Colab のシークレットとして追加（推奨）するか、以下で直接設定してください。


In [ ]:
import os
from google.colab import userdata

try:
  os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
except userdata.SecretNotFoundError:
  print(
      'GOOGLE_API_KEY not found in Colab secrets. You can still run the'
      ' notebook, but the sections using GenaiModel will fail.'
  )

In [ ]:
import asyncio
import dataclasses
import enum

import dataclasses_json
from genai_processors import content_api
from genai_processors import processor
from genai_processors import streams
from genai_processors.core import constrained_decoding
from genai_processors.core import genai_model
from google.genai import types as genai_types
from IPython.display import Markdown, display
import nest_asyncio

nest_asyncio.apply()  # Needed to run async loops in Colab

ProcessorPart = content_api.ProcessorPart

## 2. 🎬 データ構造を定義する

まず、モデルに生成させたい Python のデータ構造（`dataclasses` と `enums`）を定義します。これらは望ましい出力の「スキーマ」として機能します。

**注意:** `dataclasses` では、自動 JSON シリアライズ/デシリアライズを有効にするため `@dataclasses_json.dataclass_json` デコレータが必須です。


In [ ]:
class Genre(enum.StrEnum):
  """Enum for movie genres."""

  SCI_FI = "Science Fiction"
  FANTASY = "Fantasy"
  ACTION = "Action"
  COMEDY = "Comedy"
  DRAMA = "Drama"


@dataclasses_json.dataclass_json
@dataclasses.dataclass(frozen=True)
class Actor:
  """Represents a single actor."""

  name: str
  birth_year: int


@dataclasses_json.dataclass_json
@dataclasses.dataclass(frozen=True)
class Movie:
  """Represents a movie with its details."""

  title: str
  release_year: int
  genre: Genre
  lead_actors: list[Actor]

## 3. 🧩 カスタム Dataclass をモダリティとして扱う

このライブラリは、あなた独自のデータ型をテキストや画像と同様の第一級モダリティとして扱えるよう設計されています。`ProcessorPart` に dataclass のインスタンスを簡単に詰め込み、後から取り出せます。

これは、パイプライン内でプロセッサ間を構造化データで受け渡すのに便利です。内部的には `ProcessorPart` はオブジェクトを JSON 文字列として保持するため、テキストを想定するモデルとも互換性があります。


In [ ]:
# 1. Create an instance of our dataclass.
movie_instance = Movie(
    title="The Matrix",
    release_year=1999,
    genre=Genre.SCI_FI,
    lead_actors=[Actor(name="Keanu Reeves", birth_year=1964)],
)

# 2. Pack it into a ProcessorPart.
part = ProcessorPart.from_dataclass(dataclass=movie_instance)

print("The underlying representation is just JSON:")
print(part.text)
print("\n---\n")

# 3. Unpack it back into a Python object.
unpacked_movie = part.get_dataclass(Movie)

print(f"Unpacked a '{type(unpacked_movie).__name__}' object:")
print(unpacked_movie)

# They are identical.
assert movie_instance == unpacked_movie

## 4. 🪄 モデルからの構造化出力を自動化

データの詰め込み/取り出しは便利ですが、真価は言語モデルから直接、構造化オブジェクトを得られる点です。

`GenaiModel`（および `OllamaModel`）では、モデル設定の `response_schema` に `dataclass` や `enum` を指定するだけで、ライブラリが自動的に以下を行います:

1.  モデルに対して、スキーマに一致する JSON レスポンスを生成するよう指示する。
2.  入力される JSON ストリームをパースする。
3.  あなたの dataclass インスタンスを内包した `ProcessorPart` を逐次生成する。


### 例: 単一オブジェクトを生成

モデルに映画を考案させ、その結果を構造化された `Movie` オブジェクトとして直接受け取ってみます。


In [ ]:
# Configure the model to use our Movie dataclass as the response schema.
structured_movie_model = genai_model.GenaiModel(
    api_key=os.getenv("GOOGLE_API_KEY"),
    model_name="gemini-1.5-flash",
    generate_content_config=genai_types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Movie,
        temperature=1.0,
    ),
)

prompt = (
    "Invent a plausible but fictional sci-fi movie. It should be a completely"
    " new concept."
)
output_parts = processor.apply_sync(structured_movie_model, [prompt])

movie_instance = output_parts[0].get_dataclass(Movie)
print(
    f"The model generated a '{type(movie_instance).__name__}' object"
    " directly!\n"
)
print(movie_instance)

### 例: 複数オブジェクト（リスト）を生成

リストを扱う場合はさらに強力です。ターゲットスキーマを `list`（例: `list[Movie]`）として指定すると、モデルプロセッサは JSON 配列をパースし、**各アイテムを独立した `ProcessorPart` として順次出力** します。

これにより、後続のプロセッサが各アイテムを個別かつ並行に処理できるパイプラインを簡単に構築できます。


In [ ]:
# This time, the schema is a list of movies.
movie_list_model = genai_model.GenaiModel(
    api_key=os.getenv("GOOGLE_API_KEY"),
    model_name="gemini-1.5-flash",
    generate_content_config=genai_types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=list[Movie],
    ),
)

prompt = "Recommend two classic fantasy movies from the 1980s."
output_parts = processor.apply_sync(movie_list_model, [prompt])

print(
    f"The model returned {len(output_parts)} separate parts, one for each"
    " movie:"
)
for i, part in enumerate(output_parts):
  movie = part.get_dataclass(Movie)
  print(f"  Part {i+1}: {movie.title}")

## 5. 🧑‍🔬 構造化データでパイプラインを組む

モデルから確実に `Movie` オブジェクトを得られるようになったので、これをパイプラインで活用しましょう。今回のエージェントは次を行います:

1.  ユーザーのリクエスト（映画のおすすめ）を受け取る。
2.  `list[Movie]` を返すように設定した `GenaiModel` を使い、各 `Movie` を個別のパートとして受け取る。
3.  各 `Movie` を受け取り、きれいな Markdown サマリに整形するカスタム `PartProcessor` を連結する。
4.  整形したサマリを表示する。


In [ ]:
@processor.part_processor_function
async def format_movie_summary(
    part: content_api.ProcessorPart,
) -> str:
  """Takes a ProcessorPart containing a Movie and yields a Markdown string."""
  movie = part.get_dataclass(Movie)  # Unpack the movie object from the part.
  if not movie:
    return  # Ignore any parts that aren't Movies.

  actor_list = ", ".join([actor.name for actor in movie.lead_actors])
  summary = (
      f"### {movie.title} ({movie.release_year})\n"
      f"**Genre**: {movie.genre.value}\n"
      f"**Starring**: {actor_list}\n"
      "---"
  )
  yield summary


movie_recommender_model = genai_model.GenaiModel(
    api_key=os.getenv("GOOGLE_API_KEY"),
    model_name="gemini-1.5-flash",
    generate_content_config=genai_types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=list[Movie],
    ),
)

recommendation_agent = movie_recommender_model + format_movie_summary

prompt = "Recommend three classic fantasy movies from the 1980s."

display(Markdown(f"**User Prompt:** *{prompt}*\n"))
display(Markdown("## Recommendations"))
async for result_part in recommendation_agent(
    processor.stream_content([prompt])
):
  display(Markdown(result_part.text))

## 6. ⚙️ 発展: 仕組みをより詳細に制御する

多くの用途では、上で示した自動処理だけで十分です。ただし高度なシナリオでは、処理をより直接的に制御したくなることもあります。


### 自動パースを無効化する

`response_schema` でモデルの出力を誘導しつつも、dataclass にパースせずに生の JSON 文字列を受け取りたい場合は、モデルプロセッサのコンストラクタで `stream_json=True` を設定します。これにより自動パースが無効化されます。


In [ ]:
# This model will still request JSON from the API, but won't parse it.
raw_json_model = genai_model.GenaiModel(
    api_key=os.getenv("GOOGLE_API_KEY"),
    model_name="gemini-1.5-flash",
    generate_content_config=genai_types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Movie,  # The API is still guided by this.
    ),
    stream_json=True,  # This is the key to disable parsing.
)

prompt = "Invent a plausible but fictional fantasy movie."
output_parts = processor.apply_sync(raw_json_model, [prompt])

raw_json_string = content_api.as_text(output_parts)
print("Got the raw JSON string from the model:\n")
print(raw_json_string)

### パーサの直接利用

この自動処理は `constrained_decoding.StructuredOutputParser` というプロセッサで実現されています。通常は直接使う必要はありませんが、モデルプロセッサ以外のソースから生の JSON ストリームを受け取り、それを型付き `ProcessorPart` に変換したい場合に便利です。


In [ ]:
# The StructuredOutputParser needs to know the target type.
json_parser = constrained_decoding.StructuredOutputParser(Movie)

json_input_stream = [
    ProcessorPart(
        '{"title": "The Matrix", "release_year": 1999, "genre": "Science'
        ' Fiction", '
    ),
    ProcessorPart(
        '"lead_actors": [{"name": "Keanu Reeves", "birth_year": 1964}]}'
    ),
]

output_parts = processor.apply_sync(json_parser, json_input_stream)
movie_instance = output_parts[0].get_dataclass(Movie)

print(f"Successfully parsed a '{type(movie_instance).__name__}' instance:")
print(movie_instance)

## 7. 🚀 次のステップ

GenAI Processors で構造化データをシームレスに扱う方法を学びました。これは信頼性と予測可能性の高い AI アプリケーションを作る上での重要なテクニックです。

さらに学ぶには、以下のノートブックも参照してください:

*   [**Processor Introduction**](https://colab.research.google.com/github/google/genai-processors/blob/main/notebooks/processor_intro.ipynb): ライブラリの中核概念を基礎から学べます。
*   [**Create Your Own Processor**](https://colab.research.google.com/github/google/genai-processors/blob/main/notebooks/create_your_own_processor.ipynb): 複雑で多段な AI パイプラインを作るためのカスタムプロセッサの作り方を学べます。
